In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# 1. Kütüphaneleri İçe Aktar
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import pandas as pd
import os
import cv2

# 2. Veri Seti Yollarını Tanımla (Güncellenmiş Temiz Yol)
BASE_DIR = '/kaggle/input/datasets/coder98/emotionpain'

# Klasör yapısına göre etiket ve görüntü yolları:
FACS_DIR = os.path.join(BASE_DIR, 'Frame_Labels', 'Frame_Labels', 'FACS')
PSPI_DIR = os.path.join(BASE_DIR, 'Frame_Labels', 'Frame_Labels', 'PSPI')
IMG_DIR = os.path.join(BASE_DIR, 'Images', 'Images')

# 3. PSPI Etiketlerini Yükle ve İşle
def load_pspi_labels(pspi_dir):
    labels = {}
    for subject in os.listdir(pspi_dir):
        subject_path = os.path.join(pspi_dir, subject)
        if os.path.isdir(subject_path):
            for file in os.listdir(subject_path):
                if file.endswith('.txt'):
                    file_path = os.path.join(subject_path, file)
                    try:
                        df = pd.read_csv(file_path, header=None)
                        # Her satır bir kareye karşılık gelir
                        labels[file] = df[0].values  # PSPI skorları
                    except Exception as e:
                        print(f"Hata: {file_path}, {e}")
    return labels

pspi_labels = load_pspi_labels(PSPI_DIR)
print(f"Toplam {len(pspi_labels)} dosya yüklendi.")

# 4. Görüntü Yollarını ve Etiketleri Eşleştir
image_paths = []
image_labels = []

for subject in os.listdir(IMG_DIR):
    subject_path = os.path.join(IMG_DIR, subject)
    if os.path.isdir(subject_path):
        for seq in os.listdir(subject_path):
            seq_path = os.path.join(subject_path, seq)
            if os.path.isdir(seq_path):
                for img_file in os.listdir(seq_path):
                    if img_file.endswith('.png'):
                        img_path = os.path.join(seq_path, img_file)
                        # Dosya adından etiket anahtarını çıkar
                        label_key = img_file.replace('.png', '')
                        # PSPI skorunu bul (örnek eşleştirme)
                        for key in pspi_labels:
                            if subject in key:
                                try:
                                    idx = int(img_file.split('aff')[-1].replace('.png', ''))
                                    if idx < len(pspi_labels[key]):
                                        pspi_score = pspi_labels[key][idx]
                                        # İkili sınıflandırma: 0 = ağrı yok, 1 = ağrı var
                                        label = 0 if pspi_score == 0 else 1
                                        image_paths.append(img_path)
                                        image_labels.append(label)
                                except:
                                    pass
                                break

print(f"Toplam {len(image_paths)} görüntü etiketlendi.")

# 5. Veri Setini Train/Test Olarak Ayır (Hasta Bazlı)
from sklearn.model_selection import train_test_split
# Hasta ID'lerini çıkar
subject_ids = [p.split('/')[-3] for p in image_paths]
unique_subjects = list(set(subject_ids))
train_subjects, test_subjects = train_test_split(unique_subjects, test_size=0.2, random_state=42)

train_indices = [i for i, s in enumerate(subject_ids) if s in train_subjects]
test_indices = [i for i, s in enumerate(subject_ids) if s in test_subjects]

X_train = [image_paths[i] for i in train_indices]
y_train = [image_labels[i] for i in train_indices]
X_test = [image_paths[i] for i in test_indices]
y_test = [image_labels[i] for i in test_indices]

# 6. Görüntü Yükleme ve Ön İşleme Fonksiyonu
def load_and_preprocess_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0  # Normalizasyon
    return img, label

# 7. tf.data.Dataset Oluştur
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_ds = test_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(32).prefetch(tf.data.AUTOTUNE)

# 8. Model Mimarisi (MobileNetV2 Transfer Learning)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Başlangıçta dondur

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
output = Dense(1, activation='sigmoid')(x)  # İkili sınıflandırma

model = Model(inputs=base_model.input, outputs=output)

# 9. Modeli Derle
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 10. Sınıf Dengesizliği için Ağırlıklı Kayıp
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

# 11. Modeli Eğit
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    class_weight=class_weights
)

# 12. Sonuçları Görselleştir
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('Model Doğruluğu')
plt.xlabel('Epoch')
plt.ylabel('Doğruluk')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Eğitim Kaybı')
plt.plot(history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('Model Kaybı')
plt.xlabel('Epoch')
plt.ylabel('Kayıp')
plt.legend()

plt.show()

# 13. Test Seti Değerlendirmesi
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Doğruluğu: {test_acc:.4f}")
print(f"Test Kaybı: {test_loss:.4f}")

Toplam 0 dosya yüklendi.
Toplam 0 görüntü etiketlendi.


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.